# Reference · Day 5 studio — the number you would actually report

**Not a marking key.** You chose your own columns and your own transformation, so your
numbers are not meant to match these. What is worth comparing is the *shape*: whether
your transformation moved your three metrics in the same direction, and whether you
could say which one you would hand a client.

The studio asked for one thing that sounds easy and is not: **beat the baseline on MAE
in dollars.** Not R-squared. The rest of this notebook is what happens when you take
that seriously — including the case where a transformation makes one metric better and
another dramatically worse, and the reason turns out to be a single house.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ames = load("ames.csv")
y = ames["SalePrice"]

NUMERIC = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF", "Lot_Area"]
CATEGORICAL = ["Neighborhood", "Central_Air"]
X = ames[NUMERIC + CATEGORICAL]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=764)
print(f"{len(X_train):,} training houses, {len(X_test)} held out")

## One scoreboard, used for everything

Three metrics, computed the same way every time, always **in dollars on the held-out
set.** Writing this once is the point: if each row were scored by its own block of
copied code, a difference between two rows could be a difference in the code.

In [ ]:
rows = {}


def score(label, pred, truth=None):
    truth = y_test if truth is None else truth
    rows[label] = (r2_score(truth, pred),
                   root_mean_squared_error(truth, pred),
                   mean_absolute_error(truth, pred))
    r2, rmse, mae = rows[label]
    print(f"  {label:<30}R2 {r2:>7.3f}   RMSE ${rmse:>9,.0f}   MAE ${mae:>9,.0f}")


def board():
    print(f"\n  {'':<30}{'R2':>7}{'RMSE $':>13}{'MAE $':>13}")
    for label, (r2, rmse, mae) in rows.items():
        print(f"  {label:<30}{r2:>7.3f}{rmse:>13,.0f}{mae:>13,.0f}")


def prep():
    return ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                 fill_value="Missing")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))]),
         CATEGORICAL),
    ])

## The bar: predict the training mean for every house

R-squared is ≈ 0 **by construction** — we predict the training mean and score on the
test set, so a slightly negative value just means "worse than the test set's own
mean." That is what R-squared *is*. But the average house is off by **MAE $62,530**,
and that is the number the homeowner cares about. The baseline is not a straw man; it
is the bar your model has to clear to have earned its existence.

In [ ]:
score("mean baseline", np.full(len(y_test), y_train.mean()))

## Case A — one honest pipeline, three numbers

Every preprocessing step inside the `ColumnTransformer`, fitted on training rows only.
Nothing here is new since Lab 1; what is new is that we report three numbers instead
of one and then have to choose between them.

In [ ]:
on_price = Pipeline([("prep", prep()), ("model", LinearRegression())]).fit(X_train, y_train)
score("OLS on price", on_price.predict(X_test))

**RMSE squares errors before averaging, so it is dominated by your worst misses. MAE
is what a person means by "how far off are you."** Neither is *correct* — the question
is what decision the number is for. A homeowner asking about their own house is asking
for MAE. A lender sizing a reserve against catastrophic mispricing is asking for RMSE.

## Case B — the scale is a decision too

`SalePrice` is right-skewed, so logging it is the obvious move. Fit the **same
pipeline** on `log(SalePrice)`, then transform the predictions back to dollars with
`exp` so everything is comparable.

Commit to a guess before you run the next cell. Almost everyone expects all three
metrics to improve.

In [ ]:
on_log = Pipeline([("prep", prep()), ("model", LinearRegression())]).fit(
    X_train, np.log(y_train))
back = np.exp(on_log.predict(X_test))
score("OLS on log(price), back to $", back)
board()

**It got worse. And better.** Both halves out loud, because the second one is not what
you would guess:

- **MAE improves.** Log space minimises *proportional* error — 10% on a $400k house
  counts the same as 10% on a $100k one — so typical houses land closer.
- **RMSE collapses.** Back-transforming with `exp` turns a large *positive* log
  residual into an enormous dollar error.

The next cell finds out how much of that collapse is one house.

In [ ]:
err = np.abs(back - y_test.to_numpy())
worst = int(np.argmax(err))
print(f"  worst miss: a ${y_test.iloc[worst]:,.0f} house predicted at ${back[worst]:,.0f}")

keep = np.arange(len(y_test)) != worst
print(f"\n  log-model RMSE, all {len(y_test)} houses:  "
      f"${root_mean_squared_error(y_test, back):,.0f}")
print(f"  log-model RMSE, dropping that one row: "
      f"${root_mean_squared_error(y_test.to_numpy()[keep], back[keep]):,.0f}")
print(f"  price-model RMSE for comparison:       "
      f"${root_mean_squared_error(y_test, on_price.predict(X_test)):,.0f}")

**One row out of 733 decided which model "won" on RMSE.** Drop it and the log model
beats the untransformed one on the very metric that made it look catastrophic.

⚠ That is the demonstration, and it is better than the abstract version: *your headline
number was set by one house, and you would never have known from the number.*

It also is not an argument against transforming. Check whether the improvement is
broad rather than a fluke of the middle of the distribution:

In [ ]:
top = y_test >= y_test.quantile(0.90)
for label, pred in (("on price", on_price.predict(X_test)), ("on log(price)", back)):
    print(f"  MAE within the most expensive 10%, {label:<14}"
          f"${mean_absolute_error(y_test[top], np.asarray(pred)[top.to_numpy()]):>9,.0f}")

The log model is better on absolute error **even in the top decile** — the place you
would most expect a proportional-error model to suffer. So the transformation is
genuinely helping typical houses across the range; it is the *squaring* in RMSE,
applied after an `exp` back-transform, that produces the scary number.

> ⚠ Do **not** let this land as "don't transform." Transforming is often right. The
> lesson is that **the metric and the scale have to be decided together**, and that a
> back-transform is a modelling decision with consequences of its own.

## The other road — transform a *predictor* instead

Several of you logged a skewed predictor rather than the outcome. That is a different
move with a different result, and it is worth seeing beside the first one.

In [ ]:
skewed = ames[NUMERIC].skew().sort_values(ascending=False)
print("  skew of the numeric predictors:")
for c, s in skewed.items():
    print(f"    {c:<18}{s:>7.2f}")

logged = X.copy()
logged["Lot_Area"] = np.log(logged["Lot_Area"])
Ltr, Lte, _, _ = train_test_split(logged, y, test_size=0.25, random_state=764)
score("OLS, log(Lot_Area)", Pipeline([("prep", prep()), ("model", LinearRegression())])
      .fit(Ltr, y_train).predict(Lte))
board()

`Lot_Area` has a skew of **12.8**, far worse than the outcome ever was, so this looks
like the higher-leverage move. It is not. R-squared and RMSE both improve a little —
and **MAE gets slightly worse.**

So the metrics disagree here too, in the opposite direction from Case B. Logging the
*outcome* helped MAE and wrecked RMSE; logging a skewed *predictor* helped RMSE and
nudged MAE the wrong way. They are different interventions: one changes what error the
model is minimising, the other changes the shape of a single column's relationship
with the target.

**If your studio result looked like this, you did not do it wrong.** You measured a
smaller effect than the person next to you, and the reason is which transformation you
reached for — not how carefully you worked.

## Why does a fit that ignores dollars win on dollars?

This is the part worth slowing down for, because the answer is not obvious: we fitted
on `log(SalePrice)` — a scale with no dollars in it anywhere — and it came out *ahead*
on MAE, which is measured in dollars. Lab 3 Part 2(b) asks you which mechanism
dominates for your own model, so here is where the mechanisms come from.

### Step 1 — what each fit is actually minimising

Least squares minimises the sum of squared errors **on whatever scale you hand it.**
Those are two different jobs:

In [ ]:
print("  two houses, each missed by a different amount:\n")
for price, miss in ((100_000, 20_000), (1_000_000, 200_000)):
    print(f"    ${price:>9,} house missed by ${miss:>8,}"
          f"   squared-dollar cost {miss**2:>18,.0f}"
          f"   proportional miss {miss/price:>5.0%}")
print("\n  Both are 20% wrong. In squared dollars the second costs 100x the first,")
print("  so a dollar-scale fit spends its effort on the expensive houses.")

### Step 2 — why the log scale is *proportional* error

The residual on the log scale is a **ratio**, not a difference. Write the relative
error as $e = \dfrac{\hat{y} - y}{y}$, so that $\hat{y}/y = 1 + e$, and the whole
thing falls out of one series:

$$\log \hat{y} - \log y \;=\; \log\!\left(\frac{\hat{y}}{y}\right)
  \;=\; \log\!\left(1 + \frac{\hat{y} - y}{y}\right)
  \;\approx\; \frac{\hat{y} - y}{y}$$

because $\log(1+e) = e - \frac{e^2}{2} + \frac{e^3}{3} - \cdots$, so dropping
everything after the first term costs you about $e^2/2$. So minimising squared error
in log space is approximately minimising squared **percentage** error in dollars.
Check that the approximation holds at the sizes of error we actually have — the last
column is the theoretical $e^2/2$, and it should track the one beside it:

In [ ]:
print(f"  {'ŷ/y':>7}{'e':>8}{'log(1+e)':>11}{'difference':>13}{'e²/2':>9}")
for ratio in (0.80, 0.90, 0.95, 1.00, 1.05, 1.10, 1.25, 1.50):
    e = ratio - 1
    print(f"  {ratio:>7.2f}{e:>+8.2f}{np.log(ratio):>11.4f}"
          f"{abs(np.log(ratio) - e):>13.4f}{e*e/2:>9.4f}")
print("\n  Within ±10% the two agree to about 0.005, and the error column tracks")
print("  e²/2 almost exactly — which is the tell that the expansion is behaving.")

### …and what it is doing *exactly*, once the errors are not small

"Percentage error" is what the log scale looks like when $e$ is small. What it is
doing **exactly**, at any size of error, is treating mistakes symmetrically in
**ratios** rather than in differences:

In [ ]:
print(f"  {'prediction':>24}{'relative err':>14}{'log residual':>14}")
for label, r in (("half the true price", 0.5), ("25% under", 0.75),
                 ("25% over", 1.25), ("double the true price", 2.0)):
    print(f"  {label:>24}{r - 1:>+14.2f}{np.log(r):>+14.3f}")

**Halving and doubling both cost $|0.693|$.** Log says *"you were wrong by a factor
of two"* and charges the same either way.

In dollars they are not the same at all: doubling is an error of $+y$ and halving is
$-y/2$, so once you square them, **doubling costs four times as much.** That is the
asymmetry the log scale removes, and it is why the fit stops chasing expensive
houses — being off by 2× on a mansion and 2× on a starter home become the same
mistake.

It also tells you exactly what happened to the house from earlier. A \$160,000 home
predicted at \$1,502,827 is a **9.4× over-prediction**, which is a log residual of
**+2.24** — large, but finite, and only about 14 residual SDs out. `exp()` is what
turns that into a \$1.34 million dollar error, and RMSE is what squares it.

### Step 3 — why that helps **MAE** in particular

MAE is an *unweighted* average of absolute errors, so it is dominated by the **many**
typical houses rather than the **few** expensive ones. A fit that treats every house
proportionally serves the dense middle of the distribution — and that is exactly what
MAE rewards. Split the held-out set into price deciles and look:

In [ ]:
dec = pd.qcut(y_test, 10, labels=False)
print(f"  {'decile':>7}{'median $':>11}{'OLS $':>10}{'log $':>10}{'winner':>9}")
for dd in range(10):
    m = (dec == dd).to_numpy()
    a = mean_absolute_error(y_test[m], on_price.predict(X_test)[m])
    b = mean_absolute_error(y_test[m], back[m])
    print(f"  {dd+1:>7}{np.median(y_test[m]):>11,.0f}{a:>10,.0f}{b:>10,.0f}"
          f"{('log' if b < a else 'OLS'):>9}")

The log model wins in **eight of ten deciles**, including the most expensive one —
the place you would most expect a proportional fit to struggle.

The two it loses are worth a second look. Decile 6 is not a pattern; it is the single
house from earlier:

In [ ]:
m6 = (dec == 5).to_numpy()
err6 = np.abs(back - y_test.to_numpy())[m6]
worst6 = int(np.argmax(err6))
keep6 = np.ones(m6.sum(), dtype=bool); keep6[worst6] = False
print(f"  decile 6 holds {m6.sum()} houses")
print(f"    log MAE, all of them            ${err6.mean():>9,.0f}")
print(f"    the worst one: ${y_test.to_numpy()[m6][worst6]:,.0f} predicted at "
      f"${back[m6][worst6]:,.0f}")
print(f"    log MAE without that one house  ${err6[keep6].mean():>9,.0f}")
print(f"    OLS MAE on the same houses      "
      f"${np.abs(on_price.predict(X_test) - y_test.to_numpy())[m6][keep6].mean():>9,.0f}")

So one row out of 733 is the entire reason decile 6 looks like a loss. Measured on the
quantity the log fit is actually optimising — percentage error — it is not close:

In [ ]:
for label, pred in (("OLS on price", on_price.predict(X_test)), ("OLS on log(price)", back)):
    pe = np.abs(pred - y_test.to_numpy()) / y_test.to_numpy()
    print(f"  {label:<20} median {np.median(pe):>6.1%}   mean {pe.mean():>6.1%}")

### Step 4 — and the log model is making a *claim*, not just rescaling

Logging does something beyond squashing a skew. Because `log(a·b) = log a + log b`, a
linear model on the log scale is a **multiplicative** model on the dollar scale:

$$\log y = \beta_0 + \beta_1 x_1 \quad\Longrightarrow\quad
  y = e^{\beta_0}\cdot e^{\beta_1 x_1}$$

One more unit of $x_1$ **multiplies** the price by $e^{\beta_1}$ — a constant
percentage, not a constant number of dollars. That is a substantive claim about
housing, and it is roughly true: a second bathroom adds some percentage to a house's
value. It does not add a flat \$15,000 whether the house is \$90,000 or \$900,000.

So the log model fits better in the middle because its *shape* is closer to right,
not because of a numerical trick.

### Step 5 — the back-transform bias, and why it is smaller than you are told

You will read that `exp()` of a mean log is the **geometric** mean, which sits below
the arithmetic mean, so back-transforming under-predicts. That is true, and the size
of it is routinely overstated:

In [ ]:
resid = np.log(y_train) - on_log.predict(X_train)
sigma = resid.std()
print(f"  geometric mean of the test prices  ${np.exp(np.log(y_test).mean()):>10,.0f}")
print(f"  arithmetic mean of the test prices ${y_test.mean():>10,.0f}")
print(f"    -> the naive worry is a shortfall of "
      f"{1 - np.exp(np.log(y_test).mean())/y_test.mean():.0%}\n")
print(f"  residual SD on the log scale  sigma = {sigma:.3f}")
print(f"  bias factor exp(sigma^2 / 2)        = {np.exp(sigma**2/2):.4f}"
      f"   -> {np.exp(sigma**2/2)-1:.1%}\n")
print(f"  mean prediction, OLS on dollars    ${on_price.predict(X_test).mean():>10,.0f}")
print(f"  mean prediction, OLS on log        ${back.mean():>10,.0f}")
print(f"  actual mean of the test prices     ${y_test.mean():>10,.0f}")

**The bias depends on the spread of the *residuals*, not the spread of the prices.**
Once the model explains most of the variation there is very little left to create a
gap — here the correction is about 1%, not the 9% the geometric-mean argument
suggests. The fix has a name (Duan's smearing estimator) and on this data you would
struggle to see it.

### Putting it together

| | |
|---|---|
| **MAE improves** | the log fit optimises proportional error, and MAE is dominated by the typical houses where that pays |
| **RMSE collapses** | `exp()` turns one large positive log residual into an enormous dollar error, and RMSE squares it |
| **the back-transform bias** | real, and on this data about 1% — much smaller than the usual warning |

Those first two are the mechanisms Lab 3 Part 2(b) asks about. Which one dominates
depends on your columns and your split, so **measure it rather than quoting this.**

## What you would say, and what you would refuse to say

| | |
|---|---|
| **"Typically off by about $20,500"** | ✅ Defensible. MAE, in dollars, on houses the model never saw. |
| **"The model explains 80% of the variance"** | ⚠ True and nearly useless to a homeowner. It answers a question nobody asked. |
| **"Accurate to within $39,000"** | ❌ That is RMSE, and it is not what "typically off by" means — it is inflated by a handful of houses. |
| **"The log model is worse"** | ❌ On RMSE, because of one row. On MAE it is better everywhere, including the top decile. |

## Check your own notebook against this

Not "did I get these numbers". These:

| | |
|---|---|
| **1** | Did you compare everything **in dollars**? An R-squared computed on the log scale is not comparable to one in dollars. |
| **2** | Did you beat the baseline **on the metric you chose**, rather than switching metrics once it lost? |
| **3** | When two metrics disagreed, did you go and **find out why**, rather than picking the flattering one? |
| **4** | Can you say in one sentence which number you would give a homeowner, and why that one? |
| **5** | Does the notebook survive **Restart & Run All**? |

If 3 is a no, that is the one to fix — it is the whole meeting, and Lab 3 Part 2 is
assessing exactly it.

⚠ **You will be asked for one number.** The work is deciding which, and being able to
say what choosing it hides.